# Data Wrangling Group Project — UK IT Job Market Analysis

Dataset: `adzuna_export_16032026.xlsx`

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
import pandas as pd
import numpy as np
import re
import ast
import requests
import json
from datetime import datetime

pd.set_option('display.max_columns', None)

df_raw = pd.read_excel('/content/drive/MyDrive/DATA WRANGLING PROJECT/adzuna_export_16032026.xlsx')
print(df_raw.shape)

(2500, 22)


---
## Part 1 — Data Audit

### 1. Size

In [36]:
print(f'Rows: {df_raw.shape[0]}')
print(f'Columns: {df_raw.shape[1]}')

Rows: 2500
Columns: 22


The dataset has **2500 rows and 22 columns**.

### 2. Column descriptions

I visited adzuna.co.uk and the Adzuna API docs to understand what each field actually represents:

- **redirect_url** — tracking URL that sends users to the original job posting on Adzuna
- **description** — full job ad text as written by the employer (sometimes truncated)
- **salary_min** — lower end of advertised salary in GBP per year; can be Adzuna-predicted
- **salary_max** — upper end of advertised salary in GBP per year; can be Adzuna-predicted
- **salary_is_predicted** — 0/1 flag: 1 means Adzuna estimated the salary rather than taking it from the ad
- **\_\_CLASS\_\_** — internal API class name for the response object (always the same value)
- **id** — unique integer ID that Adzuna assigns to each listing
- **adref** — JWT token used for tracking ad clicks, not useful for analysis
- **created** — timestamp when the listing was indexed by Adzuna (ISO 8601 string)
- **title** — job title as stated by the employer
- **latitude** — geographic coordinate of the job location; missing when location is too vague
- **longitude** — same as above
- **category.\_\_CLASS\_\_** — internal API class name for the category object
- **category.label** — human-readable category name shown on Adzuna (e.g. 'IT Jobs')
- **category.tag** — URL slug used in search (e.g. 'it-jobs')
- **company.display_name** — employer name as shown on the listing
- **company.\_\_CLASS\_\_** — internal API class name for the company object
- **location.area** — hierarchical location list from broadest to most specific (stored as a string)
- **location.\_\_CLASS\_\_** — internal API class name for the location object
- **location.display_name** — readable location string shown on the listing (e.g. 'London, UK')
- **contract_time** — full_time or part_time; often missing
- **contract_type** — permanent or contract; often missing

### 3. Missing values

In [37]:
missing = df_raw.isnull().sum()
pct = (missing / len(df_raw) * 100).round(1)
result = pd.DataFrame({'missing_count': missing, 'missing_%': pct})
result[result['missing_count'] > 0].sort_values('missing_%', ascending=False)

,missing_count,missing_%
contract_type,1370,54.8
contract_time,1055,42.2
latitude,862,34.5
longitude,862,34.5
company.display_name,14,0.6
salary_max,5,0.2
salary_min,2,0.1


Columns with more than 30% missing (need special attention):
- **contract_type**: 54.8% missing
- **contract_time**: 42.2% missing
- **latitude / longitude**: 34.5% missing each

### 4. Data types

In [38]:
df_raw.dtypes

,0
redirect_url,object
description,object
salary_min,float64
__CLASS__,object
salary_is_predicted,int64
id,int64
latitude,float64
created,object
adref,object
salary_max,float64


Issues I spotted:
- `created` is stored as `object` (string) but it should be a datetime
- `location.area` looks like a list (e.g. `['UK', 'London', ...]`) but is stored as a plain string — needs to be parsed
- `salary_is_predicted` is stored as int64 but is really a boolean flag

### 5. Which columns can be dropped? (__CLASS__ columns)

In [39]:
class_cols = [c for c in df_raw.columns if '__CLASS__' in c]
for col in class_cols:
    print(f'{col}: {df_raw[col].unique()}')

__CLASS__: ['Adzuna::API::Response::Job']
category.__CLASS__: ['Adzuna::API::Response::Category']
company.__CLASS__: ['Adzuna::API::Response::Company']
location.__CLASS__: ['Adzuna::API::Response::Location']


All four `__CLASS__` columns contain a single constant value across every row — they're just internal Adzuna API type labels that got included in the export. They carry zero information and can be dropped.

I'll also drop:
- `category.label` and `category.tag` — constant (we only queried the IT Jobs category)
- `adref` — JWT tracking token, not usable for analysis

### 6. Salary plausibility

In [40]:
print('salary_min:')
print(df_raw['salary_min'].describe())
print()
print('salary_max:')
print(df_raw['salary_max'].describe())

salary_min:
count      2498.000000
mean      58776.431930
std       27471.854503
min           0.000000
25%       40250.787500
50%       54248.155000
75%       70000.000000
max      228800.000000
Name: salary_min, dtype: float64

salary_max:
count      2495.000000
mean      63892.612008
std       31644.934569
min           6.000000
25%       44986.405000
50%       56940.730000
75%       75000.000000
max      280000.000000
Name: salary_max, dtype: float64


In [41]:
df_raw[df_raw['salary_min'] < 500][['title','salary_min','salary_max','description']].head(8)

,title,salary_min,salary_max,description
279,System Administrator,0.0,117395.0,System Administrator Broad Oak Based £56.44 ho...
296,Document Handler,0.0,26956.0,Exciting training and bonus opportunities are ...
527,Software Engineer,0.0,124800.0,Our client has an opportunity for a Software E...
1290,Software QA Engineer,0.0,50000.0,Excellent opportunity for a Software QA Engine...
1627,Communications Advisor,0.0,133120.0,"Our client BAE Systems, a prominent entity in ..."
1784,Billings & Data Administrator,14.0,15.0,We’re delighted to be partnering with our Horl...
1997,Product Designer,2.0,6.0,The secret ingredient? You At SimplyCook our m...
2079,Data Engineer,0.0,124800.0,Azure Data Engineer £480 per day | Inside IR35...


In [42]:
df_raw[df_raw['salary_max'] > 200000][['title','salary_min','salary_max']].head(8)

,title,salary_min,salary_max
25,Lead Architect,228800.0,238908.0
50,IP (Intellectual Property) Lead,167364.0,231584.0
433,Game Developer,28000.0,280000.0
451,Lead Product Engineer,140000.0,250000.0
601,PLM Data Analyst,130000.0,208000.0
626,Interim Head of Leadership Development,156000.0,208000.0
977,Partnership Disputes- Partner Role,150000.0,250000.0
999,Market Risk Business Analyst,182000.0,208000.0


In [43]:
def extract_salary_from_text(text):
    if not isinstance(text, str):
        return None
    m = re.search(r'£(\d[\d,]+)', text)
    if m:
        val = float(m.group(1).replace(',', ''))
        return val if val > 1000 else None
    return None

df_raw['salary_text'] = df_raw['description'].apply(extract_salary_from_text)

comp = df_raw.dropna(subset=['salary_min', 'salary_text']).copy()
comp['ratio'] = comp['salary_min'] / comp['salary_text']
comp[(comp['ratio'] < 0.5) | (comp['ratio'] > 2)][['title','salary_min','salary_text']].head(10)

,title,salary_min,salary_text
218,Data Entry Clerk - no experience necessary,25000.00,58000.0
369,"Software Developer (React) - Up to £120,000 B...",50855.25,120000.0
685,Test & Release Manager,10000.00,130000.0
1572,Group FP&A Manager,10000.00,100000.0
1614,Senior Software Engineer,41761.11,150000.0
1631,Trainee Data Admin,25000.00,58000.0
1642,Platform Engineer,51970.07,120000.0
1784,Billings & Data Administrator,14.00,24000.0


**Decision:** I'll use `salary_min` and `salary_max` as the primary salary fields because they're already structured and numeric. When `salary_min` is 0 or below £1,000 (clearly wrong for an annual IT salary), I'll try to extract a salary figure from the description text instead. If that also fails, the salary will be left as NaN. I won't use text-extracted values by default because descriptions mention hourly/daily rates in many different formats, which makes extraction unreliable.

### 7. Job title diversity

In [44]:
print('Unique raw job titles:', df_raw['title'].nunique())
print()
df_raw['title'].value_counts().head(20)

Unique raw job titles: 1983



,count
title,
Internet Crowd Worker,23
Senior Software Engineer,19
Work from Home (Researcher),18
Data Engineer,17
Work from Home Independent Contractor,16
Project Manager,16
Software Engineer,14
.NET Developer,13
Business Analyst,10


There are **1983 unique job titles** across 2500 rows — nearly one unique title per listing. The main reasons are:
- Seniority levels written in different ways: "Senior Data Scientist" vs "Data Scientist Senior" vs "Sr. Data Scientist"
- Location suffixes added to titles: "Software Engineer - London"
- Abbreviations and different casing

To make titles usable, I'd normalise them first (strip seniority prefixes/suffixes, clean up casing, remove location noise), then group them into broad categories like Data, Software Engineering, Cloud/DevOps, Cybersecurity, Management, etc. Rare or ambiguous ones go into an 'Other' bucket.

### Profiling issues summary

Here's a summary of what I found before touching anything:

- **16 duplicate listing IDs** — same job appears more than once, likely from scraper pagination overlap
- **Inconsistent title casing** — some titles are all-caps (e.g. "SDET", "SAP BTP ADMIN CONSULTANT"), most are mixed
- **Many title variants for the same role** — "Solution Architect" vs "Solutions Architect", seniority prefix/suffix variation
- **contract_type missing 54.8%**, **contract_time missing 42.2%** — a lot of listings don't specify this
- **latitude/longitude missing 34.5%** — broad or country-level locations don't get coordinates
- **salary_min = 0 in several rows** — looks like day-rate contracts where the annual salary wasn't properly calculated
- **Some salary_max values look like daily rates** (e.g. £500–£650) stored where an annual figure was expected
- **`created` is a string** — should be datetime
- **`location.area` is a list stored as a string** — needs parsing
- **4 `__CLASS__` columns with one unique value each** — API metadata artefacts, no use
- **`category.label` and `category.tag` are constant** — every row is IT Jobs since we queried a single category
- **`adref` is a JWT token** — tracking artefact, not analysable
- **56% of salaries are Adzuna-predicted** (salary_is_predicted = 1), not stated by the employer — matters for salary analysis

---
## Part 2 — Data Update via Adzuna API

**Deduplication rule:** A listing is unique based on its `id` field (Adzuna's own integer primary key). I'm not using full-row equality because the same listing can appear with slightly different salary estimates or truncated descriptions. I'm also not using title + company + location because a company can legitimately post the same role in multiple locations at the same time.

In [45]:
APP_ID  = 'ce5ae1a3'
APP_KEY = '8245fc4cfb310e5c90322b29cc9889b0'

BASE_URL = 'https://api.adzuna.com/v1/api/jobs/gb/search'

def fetch_page(page):
    url = f'{BASE_URL}/{page}'
    params = {
        'app_id': APP_ID,
        'app_key': APP_KEY,
        'results_per_page': 10,
        'category': 'it-jobs',
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get('results', [])

all_records = []
for page in range(1, 51):
    try:
        records = fetch_page(page)
        all_records.extend(records)
        if page % 10 == 0:
            print(f'page {page}/50, total so far: {len(all_records)}')
    except Exception as e:
        print(f'Error on page {page}: {e}')
        break

print(f'Total fetched: {len(all_records)}')

page 10/50, total so far: 100
page 20/50, total so far: 200
page 30/50, total so far: 300
page 40/50, total so far: 400
page 50/50, total so far: 500
Total fetched: 500


In [46]:
def flatten_record(rec):
    return {
        'redirect_url':          rec.get('redirect_url'),
        'description':           rec.get('description'),
        'salary_min':            rec.get('salary_min'),
        'salary_max':            rec.get('salary_max'),
        'salary_is_predicted':   rec.get('salary_is_predicted'),
        'id':                    str(rec.get('id')),
        'created':               rec.get('created'),
        'title':                 rec.get('title'),
        'latitude':              rec.get('latitude'),
        'longitude':             rec.get('longitude'),
        'category.label':        rec.get('category', {}).get('label'),
        'category.tag':          rec.get('category', {}).get('tag'),
        'company.display_name':  rec.get('company', {}).get('display_name'),
        'location.area':         str(rec.get('location', {}).get('area', [])),
        'location.display_name': rec.get('location', {}).get('display_name'),
        'contract_time':         rec.get('contract_time'),
        'contract_type':         rec.get('contract_type'),
    }

df_api = pd.DataFrame([flatten_record(r) for r in all_records])

export_date = datetime.now().strftime('%d%m%Y')
api_filename = f'adzuna_myexport_{export_date}.csv'
df_api.to_csv(api_filename, index=False)
print(f'Saved {api_filename}')
df_api.head()

Saved adzuna_myexport_26042026.csv


,redirect_url,description,salary_min,salary_max,salary_is_predicted,id,created,title,latitude,longitude,category.label,category.tag,company.display_name,location.area,location.display_name,contract_time,contract_type
0,https://www.adzuna.co.uk/jobs/land/ad/56434384...,Professional Services | NHS-Focused | UK-Wide ...,57963.98,57963.98,1,5643438455,2026-02-24T10:27:12Z,Management Accountant/Business Partner,NaN,NaN,IT Jobs,it-jobs,HAYS,"['UK', 'London']","London, UK",None,contract
1,https://www.adzuna.co.uk/jobs/land/ad/54438608...,Description Service Architect (Bid and Program...,60291.34,60291.34,1,5443860817,2025-10-12T05:59:46Z,Service Architect,51.287563,-0.767213,IT Jobs,it-jobs,Leidos,"['UK', 'South East England', 'Hampshire', 'Far...","Farnborough, Hampshire",full_time,None
2,https://www.adzuna.co.uk/jobs/land/ad/57070197...,Support Developer (EMV / Smart Card) We are lo...,50000.00,50000.00,0,5707019781,2026-04-22T10:36:00Z,Support Developer (EMV / Smart Card),53.757702,-2.703440,IT Jobs,it-jobs,Adria Solutions,"['UK', 'North West England', 'Lancashire', 'Pr...","Preston, Lancashire",None,permanent
3,https://www.adzuna.co.uk/jobs/land/ad/57072956...,Launch Your Cyber Security Career – Job Guaran...,10000.00,50000.00,0,5707295626,2026-04-22T17:24:52Z,Trainee Junior Security Consultant,NaN,NaN,IT Jobs,it-jobs,Newto Training,['UK'],UK,full_time,permanent
4,https://www.adzuna.co.uk/jobs/land/ad/57006327...,"Environments Manager 6 Month Contract Leeds, W...",85397.41,85397.41,1,5700632748,2026-04-15T22:10:21Z,Environments Manager,53.799599,-1.549120,IT Jobs,it-jobs,Fruition Group,"['UK', 'Yorkshire And The Humber', 'West Yorks...","Leeds, West Yorkshire",None,contract


In [47]:

if df_api.empty or 'id' not in df_api.columns:
    print('API data is empty or missing id column — skipping merge, using original dataset only')
    df_combined = df_raw.copy()
else:
    existing_ids = set(df_raw['id'].astype(str))
    df_new = df_api[~df_api['id'].isin(existing_ids)].copy()

    print(f'Total from API: {len(df_api)}')
    print(f'Already in original: {len(df_api) - len(df_new)}')
    print(f'New to append: {len(df_new)}')

    df_raw['id'] = df_raw['id'].astype(str)
    df_combined = pd.concat([df_raw, df_new], ignore_index=True)

print(f'Combined: {df_combined.shape}')

Total from API: 500
Already in original: 2
New to append: 498
Combined: (2998, 23)


---
## Part 3 — Cleaning & Transformation

In [48]:
try:
    df = df_combined.copy()
except NameError:
    df = df_raw.copy()
    df['id'] = df['id'].astype(str)

print(f'Working dataset: {df.shape}')

Working dataset: (2998, 23)


### 3a. Columns — keep, drop, or create

In [49]:
to_drop = [
    '__CLASS__', 'category.__CLASS__', 'company.__CLASS__', 'location.__CLASS__',
    'category.label', 'category.tag', 'adref'
]
df.drop(columns=[c for c in to_drop if c in df.columns], inplace=True)
print('Columns kept:', list(df.columns))

Columns kept: ['redirect_url', 'description', 'salary_min', 'salary_is_predicted', 'id', 'latitude', 'created', 'salary_max', 'title', 'longitude', 'company.display_name', 'location.area', 'location.display_name', 'contract_time', 'contract_type', 'salary_text']


In [50]:
df['salary_mid'] = (df['salary_min'] + df['salary_max']) / 2


def parse_area(area_str):
    try:
        parts = ast.literal_eval(area_str)
        return pd.Series({
            'location_country': parts[0] if len(parts) > 0 else None,
            'location_region':  parts[1] if len(parts) > 1 else None
        })
    except:
        return pd.Series({'location_country': None, 'location_region': None})

df[['location_country', 'location_region']] = df['location.area'].apply(parse_area)
df.head(3)

,redirect_url,description,salary_min,salary_is_predicted,id,latitude,created,salary_max,title,longitude,company.display_name,location.area,location.display_name,contract_time,contract_type,salary_text,salary_mid,location_country,location_region
0,https://www.adzuna.co.uk/jobs/land/ad/56603396...,Job Title: Supply Chain Function Support Manag...,52790.49,1,5660339668,55.954341,2026-03-10T15:18:42Z,52790.49,Supply Chain Function Support Manager,-4.933293,BAE Systems,"['UK', 'Scotland', 'Argyll & Bute', 'Dunoon', ...","Port Riddell, Dunoon",NaN,NaN,62000.0,52790.49,UK,Scotland
1,https://www.adzuna.co.uk/jobs/land/ad/56603397...,Job Title: Supply Chain Function Support Manag...,54695.48,1,5660339754,55.859699,2026-03-10T15:18:46Z,54695.48,Supply Chain Function Support Manager,-4.031733,BAE Systems,"['UK', 'Scotland', 'North Lanarkshire', 'Coatb...","Glenboig, Coatbridge",NaN,NaN,62000.0,54695.48,UK,Scotland
2,https://www.adzuna.co.uk/jobs/land/ad/56603397...,Job Title: Supplier Risk Assurance Lead Locati...,59519.35,1,5660339760,56.208482,2026-03-10T15:18:46Z,59519.35,Supplier Risk Assurance Lead,-5.379060,BAE Systems,"['UK', 'Scotland', 'Argyll & Bute', 'Lochgilph...","Ford, Lochgilphead",NaN,NaN,48500.0,59519.35,UK,Scotland


### 3b. Data types

In [51]:
df['created'] = pd.to_datetime(df['created'], utc=True, errors='coerce')
df['date_posted'] = df['created'].dt.date

for col in ['salary_min', 'salary_max', 'salary_mid']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['salary_is_predicted'] = df['salary_is_predicted'].astype(bool)

df.dtypes

,0
redirect_url,object
description,object
salary_min,float64
salary_is_predicted,bool
id,object
latitude,float64
created,"datetime64[ns, UTC]"
salary_max,float64
title,object
longitude,float64


### 3c. Missing values and duplicates

In [52]:
print(f'Duplicate ids: {df["id"].duplicated().sum()}')
df = df.drop_duplicates(subset='id', keep='first')
print(f'Rows after dedup: {len(df)}')

Duplicate ids: 17
Rows after dedup: 2981


In [53]:
df['contract_time'] = df['contract_time'].fillna('unknown')

df['contract_type'] = df['contract_type'].fillna('unknown')

df['company.display_name'] = df['company.display_name'].fillna('Unknown Company')

df.isnull().sum()[df.isnull().sum() > 0]

,0
salary_min,3
latitude,1000
salary_max,6
longitude,1000
salary_text,2513
salary_mid,6
location_region,450


### 3d. Numeric standardisation — salary

In [54]:
def extract_salary_from_text(text):
    if not isinstance(text, str):
        return np.nan
    m = re.search(r'£(\d[\d,]+)', text)
    if m:
        val = float(m.group(1).replace(',', ''))
        return val if val >= 1000 else np.nan
    return np.nan

bad_salary = df['salary_min'] < 1000
print(f'Rows with implausible salary_min: {bad_salary.sum()}')

df.loc[bad_salary, 'salary_min'] = df.loc[bad_salary, 'description'].apply(extract_salary_from_text)

df['salary_mid'] = (df['salary_min'].fillna(0) + df['salary_max'].fillna(0)) / 2
df.loc[df['salary_min'].isna() & df['salary_max'].isna(), 'salary_mid'] = np.nan

print(f'salary_min range after fix: {df["salary_min"].min():.0f} - {df["salary_min"].max():.0f}')

Rows with implausible salary_min: 20
salary_min range after fix: 1000 - 228800


### 3e. Outliers

In [55]:
Q1 = df['salary_mid'].quantile(0.25)
Q3 = df['salary_mid'].quantile(0.75)
IQR = Q3 - Q1

lower = max(Q1 - 3 * IQR, 0)
upper = Q3 + 3 * IQR

outliers = df[(df['salary_mid'] < lower) | (df['salary_mid'] > upper)]
print(f'Fence: £{lower:,.0f} - £{upper:,.0f}')
print(f'Outlier rows: {len(outliers)}')
outliers[['title', 'salary_min', 'salary_max', 'salary_mid']].head(10)

Fence: £0 - £155,490
Outlier rows: 53


,title,salary_min,salary_max,salary_mid
25,Lead Architect,228800.0,238908.0,233854.0
50,IP (Intellectual Property) Lead,167364.0,231584.0,199474.0
61,Senior Bid Writer - Construction/ FM,130000.0,195000.0,162500.0
94,Delivery Manager,143000.0,169000.0,156000.0
96,Trust & Safety Representative (SC Cleared),156000.0,166400.0,161200.0
120,Data Wrangler,156000.0,195000.0,175500.0
129,Business Intelligence Developer,182000.0,182000.0,182000.0
147,"Solution Architect - Real Estate, Yardi, Argus...",169000.0,182000.0,175500.0
198,Senior Technical Architect,162500.0,169000.0,165750.0
385,Principal Software Engineer / C# / Azure / Bac...,143000.0,169000.0,156000.0


In [56]:
df['salary_min_clean'] = df['salary_min'].clip(lower=lower, upper=upper)
df['salary_max_clean'] = df['salary_max'].clip(lower=lower, upper=upper)
df['salary_mid_clean'] = df['salary_mid'].clip(lower=lower, upper=upper)

print(f'salary_mid_clean range: £{df["salary_mid_clean"].min():,.0f} - £{df["salary_mid_clean"].max():,.0f}')

salary_mid_clean range: £3 - £155,490


### 3f. Job title normalisation and categorisation

#### Stage 1 — Normalise

In [57]:
def normalise_title(raw):
    if not isinstance(raw, str):
        return 'Unknown'
    t = raw.lower().strip()

    t = re.sub(r'\s*[-–,]\s*(london|manchester|birmingham|uk|remote|hybrid|on[- ]?site|nationwide|midlands|north|south|east|west|scotland|wales|england).*$', '', t)

    t = re.sub(r'\s*\([^)]*\)', '', t)

    t = re.sub(r'^(senior|sr\.?|lead|principal|staff|junior|jr\.?|associate|graduate|trainee|apprentice|mid[- ]?level|experienced)\s+', '', t, flags=re.IGNORECASE)

    t = re.sub(r'\s*[-–]\s*(senior|sr\.?|lead|principal|junior|jr\.?|associate|graduate|trainee|i{1,3}|iv|v|1|2|3)$', '', t, flags=re.IGNORECASE)

    t = re.sub(r'\s+', ' ', t).strip()
    return t.title()

df['title_normalised'] = df['title'].apply(normalise_title)

print(f'Unique raw titles:        {df["title"].nunique()}')
print(f'Unique normalised titles: {df["title_normalised"].nunique()}')
print()
df['title_normalised'].value_counts().head(20)

Unique raw titles:        2369
Unique normalised titles: 2135



,count
title_normalised,
Software Engineer,71
Data Engineer,34
Project Manager,30
Internet Crowd Worker,24
Software Developer,22
Network Engineer,20
Product Manager,20
Ai Engineer,19
Work From Home,19


#### Stage 2 — Categorise

In [58]:
CATEGORY_RULES = [
    ('Data & AI',      r'data scien|data analy|data engineer|machine learning|ml engineer|ai engineer|nlp|deep learning|business intel|bi analyst|analytics|data warehouse|etl'),
    ('Software Eng',   r'software engineer|software developer|\.net developer|java developer|python developer|full.?stack|front.?end|back.?end|mobile developer|ios |android|react |node\.?js|php developer|ruby'),
    ('Cloud/DevOps',   r'devops|cloud engineer|platform engineer|infrastructure|site reliability|\bsre\b|kubernetes|terraform|\baws\b|azure architect|\bgcp\b|devsecops'),
    ('Cybersecurity',  r'security|cyber|penetration|soc analyst|infosec|vulnerability|ethical hack|\bcissp\b|\bciso\b'),
    ('ERP/SAP',        r'\bsap\b|\boracle\b|\berp\b|dynamics 365|salesforce'),
    ('QA/Testing',     r'\bqa\b|quality assurance|test engineer|tester|\bsdet\b|automation test'),
    ('Management',     r'manager|director|head of|\bvp\b|vice president|\bcto\b|\bcio\b|\bcpo\b|delivery lead|programme manager|product owner|scrum master|agile coach'),
    ('Network/IT Ops', r'network engineer|network admin|sysadmin|system admin|helpdesk|it support|service desk|it technician|linux admin'),
]

def categorise(title):
    t = title.lower()
    for category, pattern in CATEGORY_RULES:
        if re.search(pattern, t):
            return category
    return 'Other'

df['title_category'] = df['title_normalised'].apply(categorise)

print(df['title_category'].value_counts())
print(f'\nCoverage: {(df["title_category"] != "Other").mean() * 100:.1f}% categorised')

title_category
Other             1351
Management         510
Software Eng       362
Data & AI          219
Cybersecurity      151
Cloud/DevOps       146
Network/IT Ops     113
ERP/SAP             72
QA/Testing          57
Name: count, dtype: int64

Coverage: 54.7% categorised


In [59]:
df[df['title_category'] == 'Other']['title_normalised'].value_counts().head(20)

,count
title_normalised,
Internet Crowd Worker,24
Work From Home,19
Business Analyst,18
Work From Home Independent Contractor,17
Solution Architect,13
Solutions Architect,11
Data Architect,9
Systems Engineer,8
Technical Architect,7


In [60]:
def get_seniority(raw):
    t = raw.lower()
    if re.search(r'\b(senior|sr\.?|lead|principal|staff|head)\b', t):
        return 'Senior'
    elif re.search(r'\b(junior|jr\.?|associate|graduate|trainee|apprentice|entry)\b', t):
        return 'Junior'
    return 'Mid'

df['seniority'] = df['title'].apply(get_seniority)
df['seniority'].value_counts()

,count
seniority,
Mid,2011
Senior,784
Junior,186


---
## Export

In [61]:
final_cols = [
    'id', 'title', 'title_normalised', 'title_category', 'seniority',
    'company.display_name', 'location.display_name', 'location_country', 'location_region',
    'latitude', 'longitude', 'contract_time', 'contract_type',
    'salary_min_clean', 'salary_max_clean', 'salary_mid_clean', 'salary_is_predicted',
    'created', 'date_posted', 'description', 'redirect_url'
]

df_clean = df[[c for c in final_cols if c in df.columns]]
df_clean.to_csv('adzuna_cleaned.csv', index=False)
print(f'Saved adzuna_cleaned.csv — {df_clean.shape[0]} rows x {df_clean.shape[1]} columns')
df_clean.head()

Saved adzuna_cleaned.csv — 2981 rows x 21 columns


,id,title,title_normalised,title_category,seniority,company.display_name,location.display_name,location_country,location_region,latitude,longitude,contract_time,contract_type,salary_min_clean,salary_max_clean,salary_mid_clean,salary_is_predicted,created,date_posted,description,redirect_url
0,5660339668,Supply Chain Function Support Manager,Supply Chain Function Support Manager,Management,Mid,BAE Systems,"Port Riddell, Dunoon",UK,Scotland,55.954341,-4.933293,unknown,unknown,52790.49,52790.49,52790.49,True,2026-03-10 15:18:42+00:00,2026-03-10,Job Title: Supply Chain Function Support Manag...,https://www.adzuna.co.uk/jobs/land/ad/56603396...
1,5660339754,Supply Chain Function Support Manager,Supply Chain Function Support Manager,Management,Mid,BAE Systems,"Glenboig, Coatbridge",UK,Scotland,55.859699,-4.031733,unknown,unknown,54695.48,54695.48,54695.48,True,2026-03-10 15:18:46+00:00,2026-03-10,Job Title: Supply Chain Function Support Manag...,https://www.adzuna.co.uk/jobs/land/ad/56603397...
2,5660339760,Supplier Risk Assurance Lead,Supplier Risk Assurance Lead,Other,Senior,BAE Systems,"Ford, Lochgilphead",UK,Scotland,56.208482,-5.379060,unknown,unknown,59519.35,59519.35,59519.35,True,2026-03-10 15:18:46+00:00,2026-03-10,Job Title: Supplier Risk Assurance Lead Locati...,https://www.adzuna.co.uk/jobs/land/ad/56603397...
3,5660339813,Digital Solutions Analyst – Supply Chain,Digital Solutions Analyst – Supply Chain,Other,Mid,BAE Systems,"Heysham, Morecambe",UK,North West England,54.085712,-2.926601,unknown,unknown,41902.30,41902.30,41902.30,True,2026-03-10 15:18:49+00:00,2026-03-10,Job Title: Digital Solutions Analyst – Supply ...,https://www.adzuna.co.uk/jobs/land/ad/56603398...
4,5654983934,Automotive Testing_Senior Engineer_Pune,Automotive Testing_Senior Engineer_Pune,Other,Mid,"Arrow Electronics, Inc.",UK,UK,None,NaN,NaN,full_time,unknown,49632.25,49632.25,49632.25,True,2026-03-05 14:22:43+00:00,2026-03-05,Position: Automotive Testing_Senior Engineer_P...,https://www.adzuna.co.uk/jobs/land/ad/56549839...


---
## Final profiling summary

Issues found and what I did about each:

- **16 duplicate listing IDs** → dropped duplicates, kept first occurrence
- **4 `__CLASS__` columns (constant values)** → dropped
- **`category.label` / `category.tag` constant** → dropped
- **`adref` JWT token** → dropped
- **`created` stored as string** → converted to datetime64 UTC, added `date_posted` column (date only)
- **`location.area` stored as list-string** → parsed with ast.literal_eval, extracted `location_country` and `location_region`
- **`salary_is_predicted` stored as int** → converted to bool
- **`salary_min` = 0 or implausibly low** → replaced with salary extracted from description text where possible, else NaN
- **Salary outliers (likely day rates stored as annual)** → winsorised at 3x IQR bounds, stored in `salary_*_clean` columns
- **`contract_time` missing 42%** → filled with 'unknown'
- **`contract_type` missing 55%** → filled with 'unknown'
- **`company.display_name` missing 14 rows** → filled with 'Unknown Company'
- **`latitude`/`longitude` missing 35%** → left as NaN (reflects genuinely vague locations)
- **1983 unique raw job titles, inconsistent casing** → normalised (stripped seniority, location noise, standardised casing), then categorised into 9 groups: Data & AI, Software Eng, Cloud/DevOps, Cybersecurity, ERP/SAP, QA/Testing, Management, Network/IT Ops, Other

In [62]:
from google.colab import files
files.download('adzuna_cleaned.csv')
files.download(api_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>